# R Programming — Basics & Weather Data Analysis

This notebook covers:
1. Conditional statements
2. Functions
3. Loops
4. Loading and exploring `weather.json`
5. Statistical description
6. Linear regression for prediction

## Setup — Install Required Packages

Run the cell below **once** before starting. It installs all packages needed for this notebook.


In [1]:
# Install only the packages that are not available yet (safe to re-run this cell)
pkgs <- c("jsonlite", "ggplot2")
missing_pkgs <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(missing_pkgs) > 0) {
  install.packages(missing_pkgs, repos = "https://cloud.r-project.org")
}

## 1. Conditional Statements

R supports `if`, `else if`, and `else`. Conditions use standard comparison operators (`==`, `!=`, `<`, `>`, `<=`, `>=`) and logical operators (`&`, `|`, `!`).

In [2]:
temp <- 35

# if / else if / else: evaluated top to bottom; only the first true branch runs
if (temp > 40) {
  cat("Extreme heat\n")
} else if (temp > 30) {
  cat("Hot day\n")
} else if (temp > 20) {
  cat("Warm day\n")
} else {
  cat("Cool or cold\n")
}

# Scalar logical operators: AND (&), OR (|), NOT (!)
humidity <- 80
if (temp > 30 & humidity > 70) {
  cat("Hot and humid -- feels oppressive\n")
}

rain_today <- "Yes"
if (rain_today == "Yes" | temp < 10) {
  cat("Bring a jacket\n")
}

# ifelse() is vectorised: it evaluates the condition element by element
temps <- c(12, 25, 38, 5, 30)
labels <- ifelse(temps > 28, "Hot", "Not hot")
print(labels)

Hot day
Hot and humid -- feels oppressive
Bring a jacket
[1] "Not hot" "Not hot" "Hot"     "Not hot" "Hot"    


## 2. Functions

Functions are defined with `function()`. R returns the last evaluated expression automatically, or you can use `return()` explicitly.

In [3]:
# Basic function with an explicit return value
celsius_to_fahrenheit <- function(c) {
  return(c * 9 / 5 + 32)
}
cat("25°C =", celsius_to_fahrenheit(25), "°F\n")

# Default argument values (unit = "C" if none is given).
# Note: the classification thresholds are the same regardless of the unit,
# so Fahrenheit values should be converted to Celsius before calling the function.
describe_temp <- function(temp, unit = "C") {
  label <- ifelse(temp > 30, "hot", ifelse(temp > 15, "warm", "cold"))
  cat(sprintf("%.2f° %s is %s\n", temp, unit, label))
}
describe_temp(35)
describe_temp(68, unit = "F")

# A function can return several values packed into a named list
temp_stats <- function(temps) {
  list(
    mean  = mean(temps),
    sd    = sd(temps),
    min   = min(temps),
    max   = max(temps),
    range = max(temps) - min(temps)
  )
}

stats <- temp_stats(c(12, 18, 25, 33, 29, 14))
print(stats)

25°C = 77 °F
35.00° C is hot
68.00° F is hot
$mean
[1] 21.83333

$sd
[1] 8.471521

$min
[1] 12

$max
[1] 33

$range
[1] 21



## 3. Loops

R has `for`, `while`, and `repeat` loops. Use `next` to skip an iteration and `break` to exit early.

In [4]:
# --- for: iterates directly over the elements of a vector ---
days <- c("Mon", "Tue", "Wed", "Thu", "Fri")
for (day in days) {
  cat("Day:", day, "\n")
}

# for with next (skip the iteration) and break (exit the loop)
cat("\nPrinting only even numbers up to 8:\n")
for (i in 1:10) {
  if (i %% 2 != 0) 
    next   # %% is the modulo operator: skips odd numbers --- 'continue' in python
  if (i > 8) break        # termina al superar 8
  cat(i, "")
}
cat("\n")

# --- while: repeats as long as the condition is true ---
cat("\nCountdown:\n")
n <- 5
while (n > 0) {
  cat(n, "")
  n <- n - 1
}
cat("Go!\n")

# --- repeat: infinite loop that requires a manual break ---
cat("\nFirst temp above 30:\n")
temps <- c(18, 22, 28, 31, 35)
i <- 1
repeat {
  if (temps[i] > 30) {
    cat("Found:", temps[i], "at index", i, "\n")
    break
  }
  i <- i + 1
}

Day: Mon 
Day: Tue 
Day: Wed 
Day: Thu 
Day: Fri 

Printing only even numbers up to 8:
2 4 6 8 

Countdown:
5 4 3 2 1 Go!

First temp above 30:
Found: 31 at index 4 


## 4. Loading Weather Data

We use `jsonlite` to read the JSON file directly into a data frame, then inspect its structure.

> **Install required package** (run once if not already installed):
> ```r
> install.packages("jsonlite")
> ```


In [5]:
library(jsonlite)

# Load weather.json (it must be in the same directory as the notebook).
# fromJSON turns an array of JSON objects into a data.frame.
df <- fromJSON("weather.json")

# In the JSON these columns are stored as text ("6.3"), and a few values are the
# literal string "NA". as.numeric() turns those into real NA values; the coercion
# warning is expected, so it is silenced.
df$Sunshine      <- suppressWarnings(as.numeric(df$Sunshine))
df$WindGustSpeed <- suppressWarnings(as.numeric(df$WindGustSpeed))
df$WindSpeed9am  <- suppressWarnings(as.numeric(df$WindSpeed9am))

cat("Dimensions:", nrow(df), "rows x", ncol(df), "columns\n\n")

# Column names
cat("Columns:\n")
print(names(df))

# First rows of the main variables
head(df[, c("MinTemp","MaxTemp","Rainfall","Humidity9am","Humidity3pm",
            "Pressure9am","Temp9am","Temp3pm","RainTomorrow")])

Dimensions: 366 rows x 22 columns

Columns:
 [1] "MinTemp"       "MaxTemp"       "Rainfall"      "Evaporation"  
 [5] "Sunshine"      "WindGustDir"   "WindGustSpeed" "WindDir9am"   
 [9] "WindDir3pm"    "WindSpeed9am"  "WindSpeed3pm"  "Humidity9am"  
[13] "Humidity3pm"   "Pressure9am"   "Pressure3pm"   "Cloud9am"     
[17] "Cloud3pm"      "Temp9am"       "Temp3pm"       "RainToday"    
[21] "RISK_MM"       "RainTomorrow" 


,MinTemp,MaxTemp,Rainfall,Humidity9am,Humidity3pm,Pressure9am,Temp9am,Temp3pm,RainTomorrow
,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<chr>
1,8.0,24.3,0.0,68,29,1019.7,14.4,23.6,Yes
2,14.0,26.9,3.6,80,36,1012.4,17.5,25.7,Yes
3,13.7,23.4,3.6,82,69,1009.5,15.4,20.2,Yes
4,13.3,15.5,39.8,62,56,1005.5,13.5,14.1,Yes
5,7.6,16.1,2.8,68,49,1018.3,11.1,15.4,No
6,6.2,16.9,0.0,70,57,1023.8,10.9,14.8,No


In [ ]:
# str(): type and first values of each column
str(df)

# summary(): minimum, quartiles, mean and maximum of the selected variables
summary(df[, c("MinTemp","MaxTemp","Rainfall","Humidity9am","Humidity3pm","Pressure9am")])

## 5. Statistical Description

We compute descriptive statistics (mean, median, SD, variance, skewness, kurtosis) and visualise distributions and correlations with `ggplot2`.

> **Install required package** (run once if not already installed):
> ```r
> install.packages("ggplot2")
> ```


In [ ]:
library(ggplot2)

# Numeric variables of interest
num_cols <- c("MinTemp","MaxTemp","Rainfall","Evaporation",
              "Humidity9am","Humidity3pm","Pressure9am","Pressure3pm",
              "Temp9am","Temp3pm")
num_df <- df[, num_cols]

# --- Descriptive statistics with base R ---
desc_stats <- function(x) {
  x   <- x[!is.na(x)]          # ignora valores faltantes
  n   <- length(x)
  mn  <- mean(x)
  med <- median(x)
  s   <- sd(x)                 # sample standard deviation (n - 1)
  v   <- var(x)
  # Skewness and kurtosis based on standardised moments
  skew <- mean(((x - mn) / s)^3)
  kurt <- mean(((x - mn) / s)^4) - 3   # excess kurtosis (normal = 0)
  c(n = n, mean = round(mn, 2), median = round(med, 2),
    sd = round(s, 2), var = round(v, 2),
    skewness = round(skew, 3), kurtosis = round(kurt, 3))
}

# sapply returns one column per variable; t() transposes it to one row per variable
stats_table <- t(sapply(num_df, desc_stats))
print(as.data.frame(stats_table))

In [ ]:
# Histogram of MaxTemp: shows the shape of the distribution (2 °C bins)
ggplot(df, aes(x = MaxTemp)) +
  geom_histogram(binwidth = 2, fill = "steelblue", color = "white") +
  labs(title = "Distribution of MaxTemp", x = "Max Temperature (°C)", y = "Count") +
  theme_minimal()

In [ ]:
# Boxplot of MaxTemp by whether it rained the next day
ggplot(df, aes(x = RainTomorrow, y = MaxTemp, fill = RainTomorrow)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_manual(values = c("No" = "#3398c4", "Yes" = "#e07474")) +
  labs(title = "MaxTemp by Rain Tomorrow", x = "Rain Tomorrow", y = "Max Temperature (°C)") +
  theme_minimal() +
  theme(legend.position = "none")   # the colour is already encoded on the x axis

In [ ]:
# Pearson correlation matrix (uses pairwise complete observations)
cor_mat <- cor(num_df, use = "pairwise.complete.obs")

# Reshape to long format (one row per variable pair), which is what ggplot expects
cor_long <- as.data.frame(as.table(cor_mat))
names(cor_long) <- c("Var1", "Var2", "Correlation")

# Heatmap: blue = negative correlation, red = positive, white = ~0
ggplot(cor_long, aes(Var1, Var2, fill = Correlation)) +
  geom_tile(color = "white") +
  scale_fill_gradient2(low = "#162ca9", mid = "white", high = "#de1616", midpoint = 0) +
  geom_text(aes(label = round(Correlation, 2)), size = 2.8) +
  labs(title = "Correlation Matrix — Weather Variables") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

## 6. Linear Regression — Predicting MaxTemp

We build a multiple linear regression model with `lm()` to predict `MaxTemp` from `MinTemp`, `Humidity3pm`, `Pressure9am`, and `Sunshine`. The data is split 80/20 into training and test sets, and the model is evaluated with RMSE and R².

In [ ]:
# Keep only the rows with no NA in the model columns
model_df <- df[, c("MaxTemp","MinTemp","Humidity3pm","Pressure9am","Sunshine")]
model_df <- na.omit(model_df)

cat("Rows available for modelling:", nrow(model_df), "\n")

# --- Train / test split (80 / 20) ---
set.seed(42)                       # fixed seed so the split is reproducible
n         <- nrow(model_df)
train_idx <- sample(seq_len(n), size = floor(0.8 * n))
train     <- model_df[train_idx, ]
test      <- model_df[-train_idx, ]

# --- Fit the multiple linear regression model ---
model <- lm(MaxTemp ~ MinTemp + Humidity3pm + Pressure9am + Sunshine, data = train)
summary(model)

In [ ]:
# --- Evaluation on the test set (data the model has not seen) ---
predictions <- predict(model, newdata = test)

# RMSE: typical error in the same units as the variable (°C)
rmse <- sqrt(mean((test$MaxTemp - predictions)^2))

# R²: proportion of variance explained = 1 - SS_residual / SS_total
ss_res <- sum((test$MaxTemp - predictions)^2)
ss_tot <- sum((test$MaxTemp - mean(test$MaxTemp))^2)
r2     <- 1 - ss_res / ss_tot

cat(sprintf("Test RMSE : %.3f C\n", rmse))
cat(sprintf("Test R²   : %.4f\n", r2))

In [ ]:
# --- Actual vs. predicted values ---
# Points close to the red diagonal (y = x) indicate good predictions
results <- data.frame(actual = test$MaxTemp, predicted = predictions)

ggplot(results, aes(x = actual, y = predicted)) +
  geom_point(alpha = 0.5, color = "steelblue") +
  geom_abline(slope = 1, intercept = 0, color = "red", linetype = "dashed", linewidth = 1) +
  labs(
    title = "Actual vs Predicted MaxTemp",
    x = "Actual MaxTemp (°C)",
    y = "Predicted MaxTemp (°C)"
  ) +
  theme_minimal()

In [ ]:
# --- Residuals vs. fitted values ---
# A good model shows residuals scattered randomly around 0, with no pattern
results$residual <- results$actual - results$predicted

ggplot(results, aes(x = predicted, y = residual)) +
  geom_point(alpha = 0.5, color = "steelblue") +
  geom_hline(yintercept = 0, color = "red", linetype = "dashed", linewidth = 1) +
  labs(
    title = "Residuals vs Fitted",
    x = "Fitted (Predicted) MaxTemp (°C)",
    y = "Residual (°C)"
  ) +
  theme_minimal()